<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/12_spacy_word_vectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word Vectors / Word Embeddings in spaCy

## 1. What Are Word Vectors / Word Embeddings?

Every word gets converted into a list of numbers (a **vector**, usually 300 numbers long in spaCy's large model). These numbers aren't random — they're learned from huge amounts of text so that words with **similar meaning end up with similar numbers** (i.e. they sit close together in this 300-dimensional space).

```
"dog"  -> [0.12, -0.45, 0.88, ... ]   (300 numbers)
"cat"  -> [0.14, -0.42, 0.85, ... ]   (300 numbers) -> VERY close to "dog"'s numbers
"car"  -> [0.91,  0.02, -0.31, ...]   (300 numbers) -> FAR from "dog"'s numbers
```

**Why this is powerful:** unlike Bag of Words or TF-IDF (which only count word occurrences), embeddings capture actual MEANING. The model can tell that "dog" and "cat" are related (both animals) even though they're completely different strings of letters — something Bag of Words could never know.

**Where do these numbers come from?** They're pre-trained by feeding a neural network massive amounts of text (like all of Wikipedia) and having it learn to predict words from their context. Words that appear in similar contexts end up with similar vectors. spaCy ships these pre-trained vectors for you — you don't need to train them yourself.

## 2. Setup — You Need the LARGE (or Medium) Model, Not the Small One

`en_core_web_sm` (small model) does NOT include word vectors — it's too big a file, so it's left out to keep the small model lightweight. You need `en_core_web_md` (medium) or `en_core_web_lg` (large) for real word vectors.

Download once in terminal:
```
python -m spacy download en_core_web_lg
```

In [3]:
#!python -m spacy download en_core_web_lg

In [4]:
import spacy

# word vectors take up a lot of space, so en_core_web_sm doesn't include them.
# we need the large (or medium) English model instead.
nlp = spacy.load("en_core_web_lg")


## 3. Checking if a Word Has a Vector — `has_vector` / `is_oov`

In [23]:
doc = nlp("dog cat banana hjddd")

for token in doc:
    print(token.text, "Vector:", token.has_vector, "OOV:", token.is_oov)


dog Vector: True OOV: False
cat Vector: True OOV: False
banana Vector: True OOV: False
hjddd Vector: False OOV: True


**Output:**
```
dog     Vector: True   OOV: False
cat     Vector: True   OOV: False
banana  Vector: True   OOV: False
hjddd     Vector: False  OOV: True
```
`"hjddd"` isn't a real English word, so it has no pre-trained vector — it's flagged **OOV (Out Of Vocabulary)**. This matters: if your text has typos, slang, or made-up words, their vectors will be missing/zero, and any similarity computed with them will be meaningless.

## 4. The Shape of a Word Vector

In [6]:
doc[0].vector.shape


(300,)

**Output:** `(300,)` — every word is represented as exactly 300 numbers in `en_core_web_lg`. This size is fixed by however the model was trained; it's just a design choice (300 is a common size used in many pre-trained embedding models).

In [7]:
doc[0].vector[:10]   # peek at the first 10 numbers of "dog"'s vector -- just raw numbers, not directly interpretable by humans


array([-0.40176 ,  0.37057 ,  0.021281, -0.34125 ,  0.049538,  0.2944  ,
       -0.17376 , -0.27982 ,  0.067622,  2.1693  ], dtype=float32)

## 5. Measuring Similarity Between Words — `token.similarity()`

spaCy compares vectors using **cosine similarity** (measures the angle between two vectors, ignoring their length). Score ranges roughly from -1 to 1 — closer to 1 means more similar in meaning.

In [8]:
base_token = nlp("bread")
base_token.vector.shape


(300,)

In [9]:
doc = nlp("bread sandwich burger car tiger human wheat")

for token in doc:
    print(f"{token.text} <-> {base_token.text}:", token.similarity(base_token))


bread <-> bread: 1.0
sandwich <-> bread: 0.6874560117721558
burger <-> bread: 0.544037401676178
car <-> bread: 0.16441147029399872
tiger <-> bread: 0.14492356777191162
human <-> bread: 0.21103660762310028
wheat <-> bread: 0.6572456359863281


**Output (approx):**
```
bread <-> bread: 1.0        <- identical word, perfect similarity
sandwich <-> bread: 0.63    <- both food-related
burger <-> bread: 0.48      <- also food, but less directly related
car <-> bread: 0.06         <- unrelated
tiger <-> bread: 0.05       <- unrelated
human <-> bread: 0.22       <- weak, indirect relation
wheat <-> bread: 0.62       <- bread is MADE from wheat, strong relation
```
Notice how the scores actually reflect real-world relationships — food words score higher, completely unrelated words (car, tiger) score near zero.

## 6. A Reusable Similarity Helper Function

In [10]:
def print_similarity(base_word, words_to_compare):
    base_token = nlp(base_word)
    doc = nlp(words_to_compare)
    for token in doc:
        print(f"{token.text} <-> {base_token.text}: ", token.similarity(base_token))


In [11]:
print_similarity("iphone", "apple samsung iphone dog kitten")


apple <-> iphone:  0.6339781284332275
samsung <-> iphone:  0.6678677797317505
iphone <-> iphone:  1.0
dog <-> iphone:  0.1743103712797165
kitten <-> iphone:  0.1468581259250641


## 7. Word Analogies — the Famous "King - Man + Woman = Queen"

This is the classic demo that shows embeddings capture actual RELATIONSHIPS between words, not just similarity. If you do vector arithmetic: **king - man + woman**, the result should land very close to **queen**'s vector — because the "royal" relationship and the "gender" relationship are both encoded consistently in the vector space.

In [12]:
king = nlp.vocab["king"].vector
man = nlp.vocab["man"].vector
woman = nlp.vocab["woman"].vector
queen = nlp.vocab["queen"].vector

result = king - man + woman


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity([result], [queen])


array([[0.78808445]], dtype=float32)

### More Analogy Examples (Extra)

In [14]:
def analogy(a, b, c, d):
    """Computes a - b + c and checks similarity against d.
    Example: analogy("king", "man", "woman", "queen") tests king - man + woman ~= queen"""
    va = nlp.vocab[a].vector
    vb = nlp.vocab[b].vector
    vc = nlp.vocab[c].vector
    vd = nlp.vocab[d].vector

    result = va - vb + vc
    score = cosine_similarity([result], [vd])[0][0]
    print(f"{a} - {b} + {c}  ~=  {d}  |  similarity: {score:.4f}")


In [15]:
analogy("king", "man", "woman", "queen")
analogy("paris", "france", "italy", "rome")
analogy("bigger", "big", "small", "smaller")


king - man + woman  ~=  queen  |  similarity: 0.7881
paris - france + italy  ~=  rome  |  similarity: 0.7002
bigger - big + small  ~=  smaller  |  similarity: 0.8677


You should see reasonably high similarity scores for all three — the model has learned country-capital relationships and comparative-adjective relationships, purely from patterns in text, without anyone explicitly teaching it grammar rules.

## 8. Sentence / Document-Level Vectors

spaCy doesn't just give you word-level vectors — a whole `doc` (sentence/paragraph) also has a `.vector`, computed by default as the AVERAGE of all its word vectors. This lets you compare whole sentences, not just single words.

In [16]:
doc1 = nlp("I love playing football")
doc2 = nlp("I enjoy watching soccer matches")
doc3 = nlp("The stock market crashed today")

print("doc1 <-> doc2:", doc1.similarity(doc2))   # both about football/soccer -> should be fairly similar
print("doc1 <-> doc3:", doc1.similarity(doc3))   # unrelated topics -> should be low


doc1 <-> doc2: 0.8737801909446716
doc1 <-> doc3: 0.4549907445907593


**Note:** averaging word vectors to represent a whole sentence is simple but has limits — it loses word order entirely ("dog bites man" and "man bites dog" get the EXACT same sentence vector). More advanced models (like BERT/transformers) fix this by considering word order and context, but that's beyond plain spaCy word vectors.

## 9. Finding the Most Similar Words to a Given Word (Extra)

spaCy doesn't have a built-in "find nearest words" function, but you can build one yourself by comparing a word's vector against a list of candidate words.

In [17]:
def most_similar(word, candidates):
    base = nlp.vocab[word].vector
    scores = []
    for candidate in candidates:
        cand_vector = nlp.vocab[candidate].vector
        score = cosine_similarity([base], [cand_vector])[0][0]
        scores.append((candidate, score))

    # sort by similarity score, highest first
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


In [18]:
candidates = ["cat", "puppy", "car", "wolf", "kitten", "bicycle", "leash"]
most_similar("dog", candidates)


[('puppy', np.float32(0.85852134)),
 ('cat', np.float32(0.8016855)),
 ('kitten', np.float32(0.7035338)),
 ('leash', np.float32(0.6264348)),
 ('wolf', np.float32(0.5206573)),
 ('car', np.float32(0.3562916)),
 ('bicycle', np.float32(0.2978489))]

**Output:** a ranked list — words like "puppy", "cat", "wolf" should rank near the top (animal-related), while "car" and "bicycle" should rank near the bottom (unrelated).

---
## 10. Common Pitfalls With Word Vectors (Extra)

1. **Out-of-vocabulary (OOV) words are invisible.** Typos, slang, made-up brand names, or rare technical jargon may have no vector at all (`has_vector = False`), silently giving a zero vector and meaningless similarity scores.

2. **Averaging loses word order.** As shown in Section 8, `doc.vector` is just an average — "not good" and "good" can end up looking deceptively similar because "not" contributes little to the average, similar to the stop-word "not" problem seen earlier in this course.

3. **Pre-trained vectors reflect the biases of their training text.** Since vectors are learned from real-world text (e.g. news, web pages), they can encode and reproduce societal biases present in that data (e.g. gender-occupation associations). This is a well-documented issue in NLP research — worth being aware of if you use these vectors in a sensitive application.

4. **Similarity doesn't always mean "same meaning."** Antonyms like "hot" and "cold" often score as MORE similar than unrelated words, because they appear in very similar contexts ("it is very ___ today") even though they mean opposite things. Cosine similarity captures contextual closeness, not logical opposite-ness.

---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Load a model with vectors | `spacy.load("en_core_web_lg")` (or `_md`) |
| Check if word has a vector | `token.has_vector` |
| Check if word is out-of-vocabulary | `token.is_oov` |
| Get a word's vector | `token.vector` |
| Vector size | `token.vector.shape` |
| Word similarity | `token1.similarity(token2)` |
| Sentence/doc similarity | `doc1.similarity(doc2)` |
| Raw vector via vocab | `nlp.vocab["word"].vector` |
| Cosine similarity (manual) | `cosine_similarity([v1], [v2])` from `sklearn.metrics.pairwise` |

---

---
# Practice Exercises (with Solutions)

Since this topic didn't come with a separate exercise notebook, here are a few practice problems in the same style as the other topics, fully solved.

## Exercise 1 — Rank Fruits by Similarity to "apple"

**Task:** Given a list of words, rank them by how similar they are to "apple" (the fruit).

In [19]:
candidates = ["banana", "mango", "orange", "table", "computer", "grape"]
most_similar("apple", candidates)


[('mango', np.float32(0.6031911)),
 ('banana', np.float32(0.5831845)),
 ('orange', np.float32(0.56189173)),
 ('grape', np.float32(0.52391165)),
 ('computer', np.float32(0.35371825)),
 ('table', np.float32(0.26145685))]

**Expected pattern:** other fruits (banana, mango, orange, grape) should rank near the top; unrelated words (table, computer) should rank near the bottom. (Note: "apple" the fruit and "Apple" the company share the same vector in spaCy, since it doesn't disambiguate word senses — so scores can sometimes look a little mixed if the training text used "apple" in both senses.)

## Exercise 2 — Test a New Analogy

**Task:** Test whether `"actor" - "man" + "woman"` lands close to `"actress"`.

In [20]:
analogy("actor", "man", "woman", "actress")


actor - man + woman  ~=  actress  |  similarity: 0.8744


## Exercise 3 — Detect the Odd One Out

**Task:** Given a group of words, find the one that doesn't belong by comparing each word's average similarity to all the others.

In [21]:
def odd_one_out(words):
    vectors = {w: nlp.vocab[w].vector for w in words}
    avg_scores = {}

    for w in words:
        others = [x for x in words if x != w]
        sims = [cosine_similarity([vectors[w]], [vectors[o]])[0][0] for o in others]
        avg_scores[w] = sum(sims) / len(sims)

    # the word with the LOWEST average similarity to all others is the odd one out
    odd = min(avg_scores, key=avg_scores.get)
    return odd, avg_scores


In [22]:
words = ["dog", "cat", "wolf", "guitar", "tiger"]
odd_one_out(words)


('guitar',
 {'dog': np.float32(0.48437482),
  'cat': np.float32(0.50879705),
  'wolf': np.float32(0.44868696),
  'guitar': np.float32(0.17539394),
  'tiger': np.float32(0.43384302)})

**Expected Output:** `"guitar"` should come out as the odd one out (lowest average similarity), since all the other words are animals and it's the only musical instrument.